In [ ]:
'''
Postoperative Complications Pipeline (v6)
========================================

This script rebuilds postoperative physiologic complications from the raw ward
vitals and timeline-labeled labs/events, merges final cluster labels, and then
computes cluster-wise chi-square tests and odds ratios (with 95% CIs). It also
computes ORs for ICU extended stay (>3 days) and in-hospital death (post-op).

Inputs (expected in the working directory):
- patient_timeline_events_labeled.csv  # labs + events (e.g., OR in/out, ICU in/out, death)
- extracted_ward_vitals.csv            # ward vitals stream (S/D BP, MBP, vent, UO, etc.)
- extracted_operations.csv             # operation-level info (e.g., weight)
- model_combined_dataset_clustered_withmissingness.csv  # final cluster labels

Key definitions:
- Post-op window: from OR_OUT (minutes) to OR_OUT + 21,600 min (15 days).
- Hypotension (sustained): MAP < 65 mmHg for ≥15 continuous minutes
    -> i.e., three consecutive readings at t, t+5, t+10 minutes with
       SBP/DBP-derived MAP = (SBP + 2*DBP)/3.
- Metabolic acidosis: any pH < 7.35 AND HCO3 < 22 mmol/L in post-op window.
- Hyperlactatemia: any lactate > 2 mmol/L in post-op window.
- AKI (KDIGO 2024):
    (a) Creatinine criterion: any post-op creatinine ≥ 1.5× baseline (baseline = closest pre-op creatinine ≤ OR_OUT).
    (b) Urine-output criterion: UO rate < 0.5 mL/kg/h for > 6 continuous hours.
       Operationalization: Sum UO in consecutive 6-hour bins from OR_OUT; for
       any bin, UO/weight/6h < 0.5 → AKI_UO=1.
- Ventilated: any ward-vitals 'vent' == 1 in post-op window.

Statistics:
- Cluster vs endpoint: Pearson chi-square (2×2, no continuity correction),
  plus Cramér’s V and per-cluster positive rates.
- Odds ratio (OR) with 95% CI (Wald on log-OR). Haldane–Anscombe 0.5 correction
  applied if any cell is zero.
- ICU extended stay (>3 days; 4320 min) computed from ICU in/out.
- In-hospital death post-op = any death time ≥ OR_OUT (subject-level earliest death if op-specific missing).

Outputs (written to the working directory):
- postop_outcomes_from_scratch_v6.csv        # Flags per operation
- chi_square_physio_by_cluster_v6_rebuild.csv
- chi_square_physio_by_cluster_report_v6_rebuild.csv
- postop_complication_ORs_cluster0_vs1_v6.csv
- icu_death_ORs_cluster0_vs1_v6.csv

Usage:
    python postop_pipeline_v6.py

Dependencies: pandas, numpy, scipy (for chi2), matplotlib (optional if plotting).
'''

import math
import numpy as np
import pandas as pd
from pathlib import Path

# --- Config ---
POSTOP_WINDOW_MIN = 21_600    # 15 days in minutes
BIN_UO_MIN = 360              # 6 hours in minutes
ICU_LOS_GT3D_MIN = 3 * 1440   # 3 days (in minutes)
OR_ORIENTATION = "cluster0_vs_1"  # report OR as High-risk (cluster 0) vs Low-risk (cluster 1)

# --- Helpers ---
def std_cols(df):
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

def load_csv(path):
    return std_cols(pd.read_csv(path))

def ensure_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def haldane_or_ci(a, b, c, d):
    """
    Odds ratio (High vs Low) = (a/b)/(c/d), Wald CI on log scale.
    Apply Haldane–Anscombe 0.5 correction if any cell is zero.
    Returns: (or_est, ci_low, ci_high, correction_applied)
    """
    corr = 0.5 if min(a,b,c,d) == 0 else 0.0
    a2, b2, c2, d2 = a+corr, b+corr, c+corr, d+corr
    or_est = (a2/b2) / (c2/d2)
    se = math.sqrt(1/a2 + 1/b2 + 1/c2 + 1/d2)
    lo = math.exp(math.log(or_est) - 1.96*se)
    hi = math.exp(math.log(or_est) + 1.96*se)
    return or_est, lo, hi, int(corr > 0)

# --- Load data ---
ROOT = Path(".")
timeline = load_csv(ROOT / "patient_timeline_events_labeled.csv")
ward = load_csv(ROOT / "data/extracted_ward_vitals.csv")
ops = load_csv(ROOT / "data/extracted_operations.csv")
clusters = load_csv(ROOT / "model_combined_dataset_clustered_withmissingness.csv")

# Normalize common fields
timeline = ensure_numeric(timeline, ["subject_id","op_id","time"])
ward = ensure_numeric(ward, ["subject_id","op_id"])
# Ward time column may be 'chart_time' or 'time'
if "chart_time" in ward.columns:
    ward = ward.rename(columns={"chart_time":"time"})
ward = ensure_numeric(ward, ["time"])
ward["value"] = pd.to_numeric(ward.get("value"), errors="coerce")

ops = ensure_numeric(ops, ["subject_id","op_id"])
# Infer weight column name
weight_col = None
for cand in ["weight_kg", "weight", "preop_weight_kg", "op_weight_kg", "admission_weight_kg"]:
    if cand in ops.columns:
        weight_col = cand
        break

# Find cluster column
cluster_col = None
for cand in ["cluster","cluster_label","risk_cluster","final_cluster"]:
    if cand in clusters.columns:
        cluster_col = cand
        break
if cluster_col is None:
    raise RuntimeError("No cluster-like column found in model_combined_dataset_clustered_withmissingness.csv")

clusters = ensure_numeric(clusters, ["subject_id","op_id"])
cluster_map = (clusters[["subject_id","op_id", cluster_col]]
               .dropna(subset=["subject_id","op_id"])
               .drop_duplicates(subset=["subject_id","op_id"])
               .rename(columns={cluster_col:"cluster"}))

# --- Build anchor table (per operation) ---
anchor_metrics = ["orin_time","orout_time","icuin_time","icuout_time","discharge_time"]
anchors = timeline[timeline["metric"].isin(anchor_metrics)][["subject_id","op_id","metric","time"]]
anchor = (anchors.pivot_table(index=["subject_id","op_id"], columns="metric", values="time", aggfunc="min")
          .reset_index())
anchor.columns.name = None

# Post-op window
anchor = anchor[anchor["orout_time"].notna()].copy()
anchor["window_start"] = anchor["orout_time"].astype(float)
anchor["window_end"] = anchor["window_start"] + POSTOP_WINDOW_MIN
win = anchor[["subject_id","op_id","window_start","window_end","orout_time","icuin_time","icuout_time"]]

# --- Hypotension (sustained 15 min) from ward vitals ---
bp = ward[ward["item_name"].isin(["nibp_sbp","nibp_dbp"])].copy()
bp = bp.merge(win, on="subject_id", how="inner")
bp = bp[(bp["time"] >= bp["window_start"]) & (bp["time"] <= bp["window_end"])]
bp_agg = (bp.groupby(["subject_id","op_id","time","item_name"])["value"].mean().reset_index())
bp_wide = (bp_agg.pivot_table(index=["subject_id","op_id","time"], columns="item_name", values="value", aggfunc="first")
           .reset_index())
bp_wide.columns.name = None
bp_wide["map_calc"] = np.where(
    bp_wide[["nibp_sbp","nibp_dbp"]].notna().all(axis=1),
    (bp_wide["nibp_sbp"] + 2*bp_wide["nibp_dbp"]) / 3.0,
    np.nan
)
bp_wide["time_round"] = bp_wide["time"].round().astype("Int64")

def sustained_low_map_flag(g):
    low_times = set(g.loc[g["map_calc"] < 65, "time_round"].dropna().astype(int).tolist())
    for t in low_times:
        if (t+5 in low_times) and (t+10 in low_times):
            return 1
    return 0

hypo = (bp_wide.groupby(["subject_id","op_id"])
        .apply(sustained_low_map_flag)
        .rename("postop_hypotension_sustained15")
        .reset_index())

# --- Metabolic acidosis & Hyperlactatemia from labs in timeline ---
# timeline uses 'metric' for lab names and 'value' for measurements
labs_needed = ["ph","hco3","lacate","lactate","creatinine"]
labs = timeline[timeline["metric"].isin(labs_needed)].copy()
labs = labs.merge(win, on=["subject_id","op_id"], how="left")

postop_labs = labs[(labs["time"] >= labs["window_start"]) & (labs["time"] <= labs["window_end"])].copy()
postop_labs["value"] = pd.to_numeric(postop_labs["value"], errors="coerce")

# Handle both 'lacate' (typo) and 'lactate'
postop_labs["metric"] = postop_labs["metric"].replace({"lacate":"lactate"})

# Flags per op
def flag_met_acid(g):
    any_ph = (g.loc[g["metric"]=="ph","value"] < 7.35).any()
    any_hco3 = (g.loc[g["metric"]=="hco3","value"] < 22).any()
    return int(bool(any_ph and any_hco3))

def flag_hyperlact(g):
    any_lac = (g.loc[g["metric"]=="lactate","value"] > 2).any()
    return int(bool(any_lac))

acid = (postop_labs.groupby(["subject_id","op_id"]).apply(flag_met_acid)
        .rename("postop_metabolic_acidosis").reset_index())
hyperlac = (postop_labs.groupby(["subject_id","op_id"]).apply(flag_hyperlact)
            .rename("postop_hyperlactatemia").reset_index())

# --- AKI: creatinine baseline + UO ---
# Baseline creatinine = closest pre-op (time <= OR_OUT) per operation
preop_creat = (labs[labs["metric"]=="creatinine"]
               .merge(win[["subject_id","op_id","orout_time"]], on=["subject_id","op_id"], how="left"))
preop_creat = preop_creat[preop_creat["time"] <= preop_creat["orout_time"]].copy()
preop_creat["value"] = pd.to_numeric(preop_creat["value"], errors="coerce")

# closest to orout_time (max time <= orout_time)
pre_baseline = (preop_creat.sort_values(["subject_id","op_id","time"])
                .groupby(["subject_id","op_id"]).tail(1)
                .rename(columns={"value":"creatinine_baseline"})[["subject_id","op_id","creatinine_baseline"]])

# Post-op creatinine
post_creat = postop_labs[postop_labs["metric"]=="creatinine"][["subject_id","op_id","time","value"]].copy()
post_creat["value"] = pd.to_numeric(post_creat["value"], errors="coerce")

# Merge to compute creatinine AKI flag
cre_merged = (post_creat.merge(pre_baseline, on=["subject_id","op_id"], how="left"))
cre_merged["creatinine_aki_hit"] = np.where(
    (cre_merged["creatinine_baseline"].notna()) & (cre_merged["value"] >= 1.5*cre_merged["creatinine_baseline"]),
    1, 0
)
aki_cre = (cre_merged.groupby(["subject_id","op_id"])["creatinine_aki_hit"].max()
           .rename("aki_creatinine").reset_index())

# UO < 0.5 mL/kg/h over any contiguous 6h bin
uo = ward[ward["item_name"]=="uo"].copy()
uo = uo.merge(win, on="subject_id", how="inner")
uo = uo[(uo["time"] >= uo["window_start"]) & (uo["time"] <= uo["window_end"])].copy()
uo["bin6h"] = ((uo["time"] - uo["window_start"]) // BIN_UO_MIN).astype("Int64")

# Attach weight
if weight_col is not None:
    wmap = ops[["subject_id","op_id", weight_col]].drop_duplicates(subset=["subject_id","op_id"])
    wmap = wmap.rename(columns={weight_col:"weight_kg"})
    uo = uo.merge(wmap, on=["subject_id","op_id"], how="left")
else:
    uo["weight_kg"] = np.nan

uo_sum = (uo.groupby(["subject_id","op_id","bin6h"])["value"].sum().rename("uo_ml").reset_index())
uo_sum = uo_sum.merge(uo[["subject_id","op_id","bin6h","weight_kg"]].drop_duplicates(),
                      on=["subject_id","op_id","bin6h"], how="left")

uo_sum["uo_mlkgh"] = uo_sum["uo_ml"] / (uo_sum["weight_kg"] * 6.0)
uo_sum["uo_low_bin"] = (uo_sum["uo_mlkgh"] < 0.5).astype(int)

aki_uo = (uo_sum.groupby(["subject_id","op_id"])["uo_low_bin"].max()
          .rename("aki_urine_output").reset_index())

# Final AKI flag
aki = (aki_cre.merge(aki_uo, on=["subject_id","op_id"], how="outer").fillna(0))
aki["aki_creatinine"] = aki["aki_creatinine"].astype(int)
aki["aki_urine_output"] = aki["aki_urine_output"].astype(int)
aki["postop_aki"] = ((aki["aki_creatinine"]==1) | (aki["aki_urine_output"]==1)).astype(int)
aki = aki[["subject_id","op_id","postop_aki","aki_creatinine","aki_urine_output"]]

# --- Ventilation flag from ward vitals ---
vent = ward[ward["item_name"]=="vent"].copy()
vent = vent.merge(win, on="subject_id", how="inner")
vent = vent[(vent["time"] >= vent["window_start"]) & (vent["time"] <= vent["window_end"])]
vent["vent_on"] = (vent["value"] == 1).astype(int)
vent_flag = (vent.groupby(["subject_id","op_id"])["vent_on"].max()
             .rename("postop_ventilated").reset_index())

# --- Assemble per-operation outcomes ---
outcomes = (win[["subject_id","op_id"]]
            .drop_duplicates()
            .merge(hypo, on=["subject_id","op_id"], how="left")
            .merge(acid, on=["subject_id","op_id"], how="left")
            .merge(hyperlac, on=["subject_id","op_id"], how="left")
            .merge(aki, on=["subject_id","op_id"], how="left")
            .merge(vent_flag, on=["subject_id","op_id"], how="left")
            .fillna(0))

for col in ["postop_hypotension_sustained15","postop_metabolic_acidosis",
            "postop_hyperlactatemia","postop_aki","postop_ventilated"]:
    outcomes[col] = outcomes[col].astype(int)

# Merge cluster labels
outcomes = outcomes.merge(cluster_map, on=["subject_id","op_id"], how="left")

# Save outcomes
out_path = ROOT / "postop_outcomes_from_scratch_v6.csv"
outcomes.to_csv(out_path, index=False)

# --- Chi-square by cluster for physiologic endpoints ---
from scipy.stats import chi2_contingency

physio_eps = ["postop_hypotension_sustained15","postop_aki","postop_metabolic_acidosis",
              "postop_hyperlactatemia","postop_ventilated"]

rows = []
for ep in physio_eps:
    anal = outcomes.dropna(subset=["cluster"]).copy()
    ct = pd.crosstab(anal["cluster"], anal[ep])
    # ensure both columns
    if 0 not in ct.columns: ct[0] = 0
    if 1 not in ct.columns: ct[1] = 0
    ct = ct[[0,1]].sort_index()
    chi2, p, dof, exp = chi2_contingency(ct.values, correction=False)
    n_tot = int(ct.values.sum())
    cramer_v = math.sqrt(chi2 / n_tot) if n_tot>0 else float("nan")  # for 2×2: V == phi
    perc = (ct[1] / ct.sum(axis=1) * 100).round(2)
    for cl in ct.index:
        rows.append({
            "endpoint": ep,
            "cluster": cl,
            "n": int(ct.loc[cl].sum()),
            "n_positive": int(ct.loc[cl, 1]),
            "percent_positive": float(perc.loc[cl]),
            "chi2": float(chi2),
            "dof": int(dof),
            "p_value": float(p),
            "cramers_v": float(cramer_v),
            "n_total": n_tot,
        })

chi_path = ROOT / "chi_square_physio_by_cluster_v6_rebuild.csv"
pd.DataFrame(rows).to_csv(chi_path, index=False)

report = (pd.DataFrame(rows).groupby("endpoint")
          .apply(lambda g: pd.Series({
              "n_total": int(g["n_total"].max()),
              "chi2": round(g["chi2"].iloc[0], 4),
              "dof": int(g["dof"].iloc[0]),
              "p_value": (f"{g['p_value'].iloc[0]:.1e}" if g["p_value"].iloc[0] < 1e-4 else f"{g['p_value'].iloc[0]:.4f}"),
              "cramers_v": round(g["cramers_v"].iloc[0], 4),
              **{f"cluster_{int(r.cluster)}_n": int(r['n']) for _, r in g.iterrows()},
              **{f"cluster_{int(r.cluster)}_%pos": float(r['percent_positive']) for _, r in g.iterrows()},
          }))
          .reset_index())

report_path = ROOT / "chi_square_physio_by_cluster_report_v6_rebuild.csv"
report.to_csv(report_path, index=False)

# --- ORs (cluster 0 vs 1) for physiologic complications ---
rows_or = []
chi_df = pd.read_csv(chi_path)
chi_df = std_cols(chi_df)

for ep, grp in chi_df.groupby("endpoint"):
    # attempt direct 0/1
    recs = {int(r.cluster): r for r in grp.itertuples(index=False)}
    if not ({0,1} <= set(recs.keys())):
        keys = sorted(recs.keys())
        remap = {keys[0]:0, keys[-1]:1}
        grp["cluster_bin"] = grp["cluster"].map(remap)
        recs = {int(r.cluster_bin): r for r in grp.itertuples(index=False)}

    r0, r1 = recs[0], recs[1]
    # High-risk (cluster 0) vs Low-risk (cluster 1)
    a = int(r0.n_positive)
    b = int(r0.n - r0.n_positive)
    c = int(r1.n_positive)
    d = int(r1.n - r1.n_positive)
    or_est, lo, hi, corr = haldane_or_ci(a,b,c,d)
    p0 = a/(a+b)*100 if (a+b)>0 else float("nan")
    p1 = c/(c+d)*100 if (c+d)>0 else float("nan")
    rows_or.append({
        "endpoint": ep,
        "cluster0_highrisk_events": a, "cluster0_highrisk_nonevents": b, "cluster0_highrisk_n": a+b, "cluster0_highrisk_pct": p0,
        "cluster1_lowrisk_events": c, "cluster1_lowrisk_nonevents": d, "cluster1_lowrisk_n": c+d, "cluster1_lowrisk_pct": p1,
        "odds_ratio_cluster0_vs_cluster1": or_est, "ci_low": lo, "ci_high": hi,
        "haldane_correction": corr,
    })

or_comp_path = ROOT / "postop_complication_ORs_cluster0_vs1_v6.csv"
pd.DataFrame(rows_or).to_csv(or_comp_path, index=False)

# --- ORs for ICU extended stay and in-hospital death ---
# Build flags
df = outcomes.merge(win[["subject_id","op_id","orout_time","icuin_time","icuout_time"]], on=["subject_id","op_id"], how="left")

# ICU extended stay (>3 days)
df["icu_los_min"] = np.where(df["icuin_time"].notna() & df["icuout_time"].notna(),
                             df["icuout_time"] - df["icuin_time"], np.nan)
df["long_icu_gt3d"] = (df["icu_los_min"] > ICU_LOS_GT3D_MIN).astype(int)

# Death: grab earliest subject-level death time and compare to orout_time
death_rows = timeline[timeline["metric"].astype(str).str.contains("death", case=False, na=False)][["subject_id","op_id","time"]].copy()
subj_death = death_rows.dropna(subset=["subject_id"]).groupby("subject_id")["time"].min().rename("death_time_any").reset_index()
df = df.merge(subj_death, on="subject_id", how="left")
df["died_inhospital_postop"] = ((df["death_time_any"].notna()) &
                                (df["orout_time"].notna()) &
                                (df["death_time_any"] >= df["orout_time"])).astype(int)

rows_id = []
for ep, label in {"long_icu_gt3d":"ICU extended stay >3 days",
                  "died_inhospital_postop":"In-hospital death (post-op)"}.items():
    tmp = df[["cluster", ep]].dropna()
    # Counts for cluster 0 (high-risk) vs 1 (low-risk)
    a = int(((tmp["cluster"]==0) & (tmp[ep]==1)).sum())
    b = int(((tmp["cluster"]==0) & (tmp[ep]==0)).sum())
    c = int(((tmp["cluster"]==1) & (tmp[ep]==1)).sum())
    d = int(((tmp["cluster"]==1) & (tmp[ep]==0)).sum())
    or_est, lo, hi, corr = haldane_or_ci(a,b,c,d)
    p0 = a/(a+b)*100 if (a+b)>0 else float("nan")
    p1 = c/(c+d)*100 if (c+d)>0 else float("nan")
    rows_id.append({
        "endpoint": label,
        "cluster0_highrisk_events": a, "cluster0_highrisk_nonevents": b, "cluster0_highrisk_n": a+b, "cluster0_highrisk_pct": p0,
        "cluster1_lowrisk_events": c, "cluster1_lowrisk_nonevents": d, "cluster1_lowrisk_n": c+d, "cluster1_lowrisk_pct": p1,
        "odds_ratio_cluster0_vs_cluster1": or_est, "ci_low": lo, "ci_high": hi,
        "haldane_correction": corr,
    })

icu_death_or_path = ROOT / "icu_death_ORs_cluster0_vs1_v6.csv"
pd.DataFrame(rows_id).to_csv(icu_death_or_path, index=False)

print("Saved:")
print("-", out_path.name)
print("-", chi_path.name)
print("-", report_path.name)
print("-", or_comp_path.name)
print("-", icu_death_or_path.name)


script_path = Path("/mnt/data/postop_pipeline_v6.py")
script_path.write_text(pipeline_code)
script_path
